In [1]:
"""
兴奋-抑制神经网络 Hopf 分叉数值模拟验证
=========================================
系统方程：
  τ_E  dv_E/dt = -v_E + [M_EE*v_E - M_EI*v_I + h_E]+
  τ_I  dv_I/dt = -v_I + [M_IE*v_E - M_II*v_I + h_I]+

其中 [z]+ = max(z, 0)
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import matplotlib.gridspec as gridspec
import os
import platform

# ── 中文字体自动配置（跨平台）──
def _set_chinese_font():
    system = platform.system()
    if system == "Darwin":
        candidates = ["Arial Unicode MS", "Heiti TC", "STHeiti",
                      "PingFang SC", "Hiragino Sans GB"]
    elif system == "Windows":
        candidates = ["Microsoft YaHei", "SimHei", "SimSun"]
    else:
        candidates = ["WenQuanYi Micro Hei", "Noto Sans CJK SC",
                      "DejaVu Sans"]
    from matplotlib import font_manager
    available = {f.name for f in font_manager.fontManager.ttflist}
    for font in candidates:
        if font in available:
            plt.rcParams["font.family"] = font
            break
    plt.rcParams["axes.unicode_minus"] = False

_set_chinese_font()

OUT_DIR = './hopf_figs'
os.makedirs(OUT_DIR, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════
# 参数设定
# ═══════════════════════════════════════════════════════════════════════
M_EE = 1.5
M_EI = 1.0
M_IE = 1.2
M_II = 0.5
tau_E = 1.0
h_E   = 2.0
h_I   = 1.0

# 理论计算
Delta_M  = (1 - M_EE) * (1 + M_II) + M_EI * M_IE
tau_I_c  = (M_II + 1) / (M_EE - 1) * tau_E
omega_0  = np.sqrt(Delta_M / (tau_E * tau_I_c))
T_0      = 2 * np.pi / omega_0

print("=" * 50)
print("理论分析结果")
print("=" * 50)
print(f"  Delta_M   = {Delta_M:.4f}  (需 > 0: {'✓' if Delta_M > 0 else '✗'})")
print(f"  tau_I^c   = {tau_I_c:.4f}")
print(f"  omega_0   = {omega_0:.4f}  rad/time")
print(f"  T_0       = {T_0:.4f}  (振荡周期)")

# 平衡点（激活区内）
v_E_star = ((1 + M_II) * h_E - M_EI * h_I) / Delta_M
v_I_star = (M_IE * h_E + (1 - M_EE) * h_I) / Delta_M
print(f"\n平衡点:  v_E* = {v_E_star:.4f},  v_I* = {v_I_star:.4f}")
print(f"  (物理有效: v_E*>0 {'✓' if v_E_star>0 else '✗'}, v_I*>0 {'✓' if v_I_star>0 else '✗'})")
print("=" * 50)


# ═══════════════════════════════════════════════════════════════════════
# ODE 右端函数
# ═══════════════════════════════════════════════════════════════════════
def relu(z):
    return np.maximum(z, 0.0)

# [修改1] 删除了原先含 Bug 的 ei_ode 函数。
# 原函数 dvI 一行将 h_I 误写为 hE 参数（两者在 h_E 扫描时取值不同），
# 导致抑制性方程的外部输入错误。此处仅保留正确版本。
def ei_ode_full(t, y, tau_I, hE=h_E, hI=h_I):
    vE, vI = y
    dvE = (-vE + relu(M_EE * vE - M_EI * vI + hE)) / tau_E
    dvI = (-vI + relu(M_IE * vE - M_II * vI + hI)) / tau_I
    return [dvE, dvI]


# ═══════════════════════════════════════════════════════════════════════
# 图1：相平面 + 时间序列（τ_I 扫描）
# ═══════════════════════════════════════════════════════════════════════
# 使用具体数值标注三个 τ_I 取值，与正文保持一致
tau_I_vals = [1.5, tau_I_c, 5.4]   # 0.5*τ_Ic, τ_Ic, 1.8*τ_Ic
labels = [
    r'$\tau_I = 1.5$（稳定焦点）',
    r'$\tau_I = \tau_I^c = 3.0$（临界）',
    r'$\tau_I = 5.4$（不稳定焦点）'
]
colors = ['#2196F3', '#FF9800', '#E53935']

t_end  = 200
t_span = (0, t_end)
t_eval = np.linspace(0, t_end, 8000)

# 临界情况：截取后半段排除瞬态
t_show_start = 80

y0 = [v_E_star + 0.3, v_I_star + 0.2]

fig = plt.figure(figsize=(18, 9))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.38)

ax_phase = fig.add_subplot(gs[:, 0])
ax_ts    = [[fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[0, 2]), fig.add_subplot(gs[0, 3])],
            [fig.add_subplot(gs[1, 1]), fig.add_subplot(gs[1, 2]), fig.add_subplot(gs[1, 3])]]

sols = []
for i, (tau_I, label, color) in enumerate(zip(tau_I_vals, labels, colors)):
    sol = solve_ivp(ei_ode_full, t_span, y0, t_eval=t_eval,
                    args=(tau_I,), method='LSODA', rtol=1e-6, atol=1e-8)
    sols.append(sol)
    if i == 1:
        mask = sol.t >= t_show_start
        ax_phase.plot(sol.y[0][mask], sol.y[1][mask],
                      color=color, lw=1.4, label=label, alpha=0.85)
    else:
        ax_phase.plot(sol.y[0], sol.y[1],
                      color=color, lw=1.4, label=label, alpha=0.85)

ax_phase.plot(v_E_star, v_I_star, 'k*', ms=12, zorder=5, label='平衡点')
ax_phase.set_xlabel(r'$v_E$', fontsize=13)
ax_phase.set_ylabel(r'$v_I$', fontsize=13)
ax_phase.set_title('相平面', fontsize=13, fontweight='bold')
ax_phase.legend(fontsize=8.5, loc='upper left')
ax_phase.grid(True, alpha=0.3)

row_labels = [r'$v_E(t)$', r'$v_I(t)$']
for col_idx, (tau_I, label, color, sol) in enumerate(zip(tau_I_vals, labels, colors, sols)):
    for row_idx in range(2):
        ax = ax_ts[row_idx][col_idx]
        if col_idx == 1:
            mask = sol.t >= t_show_start
            t_plot = sol.t[mask]
            y_plot = sol.y[row_idx][mask]
        else:
            t_plot = sol.t
            y_plot = sol.y[row_idx]
        ax.plot(t_plot, y_plot, color=color, lw=1.3)
        ax.axhline([v_E_star, v_I_star][row_idx],
                   color='k', ls='--', lw=0.9, alpha=0.6)
        ax.set_xlabel('时间', fontsize=11)
        ax.set_ylabel(row_labels[row_idx], fontsize=11)
        # [修改2] 临界情况描述改为"近似等幅振荡"，避免过度断言极限环的严格存在性
        if col_idx == 1:
            title_str = f'{label}\n{row_labels[row_idx]} (后期稳态，近似等幅振荡)'
        else:
            title_str = f'{label}\n{row_labels[row_idx]}'
        ax.set_title(title_str, fontsize=8.5)
        ax.grid(True, alpha=0.3)

fig.suptitle(
    f'兴奋-抑制网络 Hopf 分叉  ($\\tau_I^c = {tau_I_c:.2f}$, '
    f'$\\omega_0 = {omega_0:.3f}$,  $T_0 = {T_0:.2f}$)',
    fontsize=14, fontweight='bold', y=1.01
)

p1 = os.path.join(OUT_DIR, 'fig1_phase_timeseries.png')
plt.savefig(p1, dpi=150, bbox_inches='tight')
print(f"\n图1 已保存：{p1}")
plt.close()


# ═══════════════════════════════════════════════════════════════════════
# 图2：特征值实部验证
# ═══════════════════════════════════════════════════════════════════════
tau_scan = np.linspace(0.5, 8.0, 500)

real_part = 0.5 * (
    (M_EE - 1) / tau_E
    - (M_II + 1) / tau_scan
)

fig2 = plt.figure(figsize=(8, 5))
plt.plot(tau_scan, real_part, lw=2)
plt.axhline(0, color='k', linestyle='--')
plt.axvline(tau_I_c, color='r', linestyle='--',
            label=fr'$\tau_I^c={tau_I_c:.2f}$')
plt.xlabel(r'$\tau_I$', fontsize=13)
plt.ylabel(r'$\mathrm{Re}(\lambda)$', fontsize=13)
plt.legend()
plt.grid(alpha=0.3)

p2 = os.path.join(OUT_DIR, 'fig2_eigenvalue_crossing.png')
plt.savefig(p2, dpi=150, bbox_inches='tight')
print(f"图2 已保存：{p2}")
plt.close()


# ═══════════════════════════════════════════════════════════════════════
# 图3：h_E 扫描 —— 边界碰撞分叉验证
# ═══════════════════════════════════════════════════════════════════════
h_E_crit   = M_EI * h_I / (1 + M_II)
h_E_crit_I = (M_EE - 1) * h_I / M_IE
h_E_scan   = np.linspace(0.0, 5.0, 200)

vE_star_scan, vI_star_scan = [], []
for hE in h_E_scan:
    vE_s = ((1 + M_II) * hE - M_EI * h_I) / Delta_M
    vI_s = (M_IE * hE + (1 - M_EE) * h_I) / Delta_M
    vE_star_scan.append(vE_s)
    vI_star_scan.append(vI_s)

vE_star_scan = np.array(vE_star_scan)
vI_star_scan = np.array(vI_star_scan)

fig3, axes3 = plt.subplots(1, 2, figsize=(12, 4.8))
plot_data = [
    (axes3[0], vE_star_scan, r'$v_E^*$', h_E_crit,   h_E_crit_I,
     f'$h_E^c = {h_E_crit:.2f}$ ($v_E^*=0$)'),
    (axes3[1], vI_star_scan, r'$v_I^*$', h_E_crit_I, h_E_crit,
     f'$h_E^{{c,I}} = {h_E_crit_I:.2f}$ ($v_I^*=0$)'),
]

for ax, vals, var_name, primary_crit, secondary_crit, crit_label in plot_data:
    physical = np.maximum(vals, 0)

    # 线性预测：仅在两临界点之间绘制（完全激活区的线性延伸参考）
    mask_between = (h_E_scan >= min(h_E_crit_I, h_E_crit)) & \
                   (h_E_scan <= max(h_E_crit_I, h_E_crit))
    ax.plot(h_E_scan[mask_between], vals[mask_between],
            '--', lw=1.5, color='#9E9E9E', label='线性预测(两临界点之间)')

    ax.plot(h_E_scan, physical, '-', lw=2.0, color='#3F51B5',
            label=f'有效解(ReLU截断): {var_name}')

    ax.axvline(primary_crit, color='r', ls='--', lw=1.5, label=crit_label)
    ax.axvline(secondary_crit, color='#FF9800', ls=':', lw=1.2,
               label=f'另一边界 = {secondary_crit:.2f}')

    ax.axhline(0, color='k', lw=0.8)
    ax.set_xlabel(r'$h_E$', fontsize=13)
    ax.set_ylabel(f'平衡点{var_name}', fontsize=12)
    ax.set_title(f'边界碰撞分析({var_name})', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8.5)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 5)

fig3.tight_layout()
p3 = os.path.join(OUT_DIR, 'fig3_hE_border_collision.png')
plt.savefig(p3, dpi=150, bbox_inches='tight')
print(f"图3 已保存：{p3}")
plt.close()

print("\n所有图像生成完毕。")

理论分析结果
  Delta_M   = 0.4500  (需 > 0: ✓)
  tau_I^c   = 3.0000
  omega_0   = 0.3873  rad/time
  T_0       = 16.2231  (振荡周期)

平衡点:  v_E* = 4.4444,  v_I* = 4.2222
  (物理有效: v_E*>0 ✓, v_I*>0 ✓)

图1 已保存：./hopf_figs/fig1_phase_timeseries.png
图2 已保存：./hopf_figs/fig2_eigenvalue_crossing.png
图3 已保存：./hopf_figs/fig3_hE_border_collision.png

所有图像生成完毕。
